# MIMIC + 6종 PPG 주 모델
손목 데이터는 주 학습에서 보류합니다. 기존 Drive 전처리 자료와 피험자 분할이 필요합니다. 원본을 다시 다운로드하지 않습니다.
Colab GPU 필수. 두 seed의 검증 결과로 후보/공통 임계값을 고정한 후 기존 시험군을 평가합니다. 기존 시험군을 재사용한 개발 평가이며 독립적인 임상/손목 검증이 아닙니다.
상세 이유: [AF_MAIN_TWO_SOURCES.md](https://github.com/dung-ho03/capdyssey/blob/main/docs/AF_MAIN_TWO_SOURCES.md)

In [ ]:
from pathlib import Path
import subprocess, sys, torch
assert torch.cuda.is_available(), 'GPU 런타임으로 변경하세요. CPU 학습은 하지 않습니다.'
from google.colab import drive
drive.mount('/content/drive')
base=Path('/content/drive/MyDrive/Capdyssey/af_integrated')
assert (base/'audits/prepared_bundle/report.json').is_file(), '전처리 검사 보고서 필요'
repo=Path('/content/capdyssey_main_two_sources')
if not repo.exists(): subprocess.run(['git','clone','https://github.com/dung-ho03/capdyssey.git',str(repo)],check=True)
subprocess.run(['git','fetch','origin','main'],cwd=repo,check=True)
subprocess.run(['git','checkout','6b0c344'],cwd=repo,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(repo/'requirements-train.txt'),'-r',str(repo/'requirements-data.txt')],check=True)
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
with (base/'main_two_sources_v1.log').open('a') as log:
    for script,extra in [('af_pulse_experiment.py',['--two-sources']),('finalize_two_sources.py',[])]:
        cmd=[sys.executable,'-u',str(repo/'scripts'/script),'--base',str(base)]+extra
        with subprocess.Popen(cmd,cwd=repo,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as proc:
            for line in proc.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
            assert proc.wait()==0, script+' 실행 실패'
print('주 모델 학습·후보 고정·기존 시험군 평가 완료')

In [ ]:
import json
from IPython.display import display, HTML
root=base/'models/main_two_sources_v1'
report=json.loads((root/'candidate/test_report.json').read_text())
print('고정 seed/임계값:',report['selection']['seed'],report['selection']['threshold'])
display(HTML('<h3>기존 시험군 개발 평가</h3><table border=1 cellpadding=8><tr><th>자료</th><th>민감도</th><th>특이도</th><th>AUC</th></tr>'+''.join(f'<tr><td>{s}</td><td>{v["sensitivity"]:.1%}</td><td>{v["specificity"]:.1%}</td><td>{v["roc_auc"]:.3f}</td></tr>' for s,v in report['by_source'].items())+'</table>'))
print('두 자료 각각 민감도90%·특이도80% 달성:',report['research_goal_met'])
print('손목 자료 성능 보장 없음. 과거에 확인한 시험 분할을 재사용한 개발 결과입니다.')